# Step 2 — Game ID Disambiguation

Match each Epic Games free-giveaway entry to its Steam App ID via the Steam Store search API and fuzzy name matching.

**Input:**  `../raw_data/epic games data.csv` (Kaggle)
**Output:** `../raw_data/steam_matches.json`

The output maps each Epic game name to a list of `[appid, steam_name]` candidates:

- single-element list → confident match
- multi-element list → ambiguous, manually resolve in Step 3
- empty list → no Steam match (F2P, delisted, DLC, or Epic-exclusive)

**Requires:** `STEAM_API_KEY` in `../.env.local`. Steam search results are cached to `raw_data/steam_search_cache.json`, so re-runs are instant and interruptions are safe.

## Sections
1. Load and clean Epic Games data
2. Search Steam Store for each game (cached)
3. Score and classify matches
4. Review match quality
5. Print unmatched games
6. Save output JSON


In [1]:
import os
import re
import json
import time
from pathlib import Path

import requests
import pandas as pd
from dotenv import load_dotenv
from rapidfuzz import fuzz

load_dotenv(Path("../.env.local") if Path("../.env.local").exists() else ".env.local")

STEAM_API_KEY = os.getenv("STEAM_API_KEY")
print(f"API key loaded: {'yes' if STEAM_API_KEY else 'NO - check .env.local'}")

API key loaded: yes


## 1. Load and clean Epic Games data

In [2]:
DATA_DIR = Path("../raw_data")
epic_csv = DATA_DIR / "epic games data.csv"

df = pd.read_csv(epic_csv)
print(f"Loaded {len(df)} rows")
df.head()

Loaded 541 rows


,serial no.,Game,Start,End,Genre
0,1,Subnautica,12-12-2018,27-12-2018,First-ever Epic free game.
1,2,Super Meat Boy,28-12-2018,10-01-2019,Second giveaway.
2,3,What Remains of Edith Finch,11-01-2019,24-01-2019,Early weekly rotation.
3,4,The Jackbox Party Pack,24-01-2019,07-02-2019,Party game bundle.
4,5,Axiom Verge,07-02-2019,21-02-2019,Metroidvania title.


In [3]:
def clean_game_name(name: str) -> str:
    """Strip trailing (repeat), (Repeat), whitespace, and normalise dashes."""
    name = str(name).strip()
    # Remove trailing (repeat) / (repeat, New Year slot) / etc.
    name = re.sub(r'\s*\(repeat[^)]*\)\s*$', '', name, flags=re.IGNORECASE).strip()
    return name

df["clean_name"] = df["Game"].apply(clean_game_name)

# Show any names that changed
changed = df[df["Game"] != df["clean_name"]][["Game", "clean_name"]]
print(f"{len(changed)} names cleaned:")
print(changed.to_string(index=False))

77 names cleaned:
                                                        Game                                          clean_name
                                            Celeste (repeat)                                             Celeste
                                Hyper Light Drifter (repeat)                                 Hyper Light Drifter
                                        Overcooked! (repeat)                                         Overcooked!
                                  Enter the Gungeon (repeat)                                   Enter the Gungeon
                                    Into the Breach (repeat)                                     Into the Breach
                                               ABZU (repeat)                                                ABZU
                                 Kingdom: New Lands (repeat)                                  Kingdom: New Lands
                                   Metro 2033 Redux (repeat)                  

## 2. Search Steam Store for each game (results cached)

In [ ]:
SEARCH_CACHE_FILE = DATA_DIR / "steam_search_cache.json"
N_CANDIDATES = 5   # top results to fetch per query
RATE_LIMIT_S = 0.4  # seconds between requests

def steam_store_search(query: str, n: int = N_CANDIDATES) -> list[dict]:
    """
    Query the Steam store search endpoint.
    Returns a list of {id, name} dicts (up to n results).
    """
    url = "https://store.steampowered.com/api/storesearch/"
    params = {"term": query, "l": "english", "cc": "US"}
    resp = requests.get(url, params=params, timeout=15)
    resp.raise_for_status()
    items = resp.json().get("items", [])
    return [{"id": item["id"], "name": item["name"]} for item in items[:n]]

# Load existing cache (allows resuming if interrupted)
if SEARCH_CACHE_FILE.exists():
    with open(SEARCH_CACHE_FILE, "r", encoding="utf-8") as f:
        search_cache: dict[str, list[dict]] = json.load(f)
    print(f"Loaded {len(search_cache)} cached queries.")
else:
    search_cache = {}

unique_names = df["clean_name"].unique().tolist()
to_fetch = [n for n in unique_names if n not in search_cache]
print(f"{len(unique_names)} unique game names | {len(to_fetch)} need fetching (~{len(to_fetch)*RATE_LIMIT_S:.0f}s)")

for i, name in enumerate(to_fetch):
    try:
        search_cache[name] = steam_store_search(name)
    except Exception as e:
        print(f"  ERROR {name!r}: {e}")
        search_cache[name] = []
    if (i + 1) % 25 == 0 or (i + 1) == len(to_fetch):
        # Save incrementally so a crash doesn't lose progress
        with open(SEARCH_CACHE_FILE, "w", encoding="utf-8") as f:
            json.dump(search_cache, f, indent=2, ensure_ascii=False)
        print(f"  {i+1}/{len(to_fetch)} fetched, cache saved.")
    time.sleep(RATE_LIMIT_S)

print("All queries cached.")

## 3. Score and classify matches

Steam's search returns results ranked by its own relevance. We use `rapidfuzz.WRatio` to score the **top result** against the query:

- **Score ≥ HIGH_THRESHOLD** → single confident match
- **Score < HIGH_THRESHOLD** → keep top `N_CANDIDATES` for manual review
- **No results returned** → empty list (not on Steam)

In [ ]:
HIGH_THRESHOLD = 88  # WRatio score above which we trust the top result as definitive

def classify_matches(epic_name: str) -> list[tuple[int, str, float]]:
    """
    Score Steam search results for `epic_name` using rapidfuzz.WRatio.
    Returns [(appid, steam_name, score), ...]:
      - single entry  → confident match
      - multiple      → needs manual review
      - empty         → not found on Steam
    """
    candidates = search_cache.get(epic_name, [])
    if not candidates:
        return []

    scored = [
        (c["id"], c["name"], fuzz.WRatio(epic_name, c["name"]))
        for c in candidates
    ]
    scored.sort(key=lambda x: x[2], reverse=True)

    best_appid, best_name, best_score = scored[0]
    if best_score >= HIGH_THRESHOLD:
        return [(best_appid, best_name, best_score)]
    else:
        return scored  # all candidates for manual review

match_results: dict[str, list[tuple[int, str, float]]] = {
    name: classify_matches(name) for name in unique_names
}

single_match = {k: v for k, v in match_results.items() if len(v) == 1}
multi_match  = {k: v for k, v in match_results.items() if len(v) > 1}
no_match     = {k: v for k, v in match_results.items() if len(v) == 0}

print(f"High-confidence single match : {len(single_match)}")
print(f"Multiple candidates (review) : {len(multi_match)}")
print(f"No match found               : {len(no_match)}")

## 4. Review match quality

In [ ]:
single_match   = {k: v for k, v in match_results.items() if len(v) == 1}
multi_match    = {k: v for k, v in match_results.items() if len(v) > 1}
no_match       = {k: v for k, v in match_results.items() if len(v) == 0}

print(f"High-confidence single match : {len(single_match)}")
print(f"Multiple candidates (review) : {len(multi_match)}")
print(f"No match found               : {len(no_match)}")

if no_match:
    print("\nUnmatched games:")
    for name in no_match:
        print(f"  - {name}")

In [ ]:
# Show all multi-candidate games for review
print("Games needing manual review:\n")
for epic_name, candidates in sorted(multi_match.items()):
    print(f"{epic_name!r}")
    for appid, steam_name, score in candidates:
        print(f"    {appid:>8}  {score:5.1f}  {steam_name}")
    print()

## 5. Print unmatched games

In [ ]:
if no_match:
    print("Games with no Steam search results (likely F2P, delisted, or Epic-exclusive):\n")
    for name in sorted(no_match):
        print(f"  - {name}")
else:
    print("All games returned at least one Steam search result.")

## 6. Save output JSON

Writes `raw_data/steam_matches.json` mapping each Epic game name to a list of candidates: `{ "Epic Game Name": [[appid, steam_name], ...] }`. See the intro cell for how to interpret single-, multi-, and empty-list entries.

Next step: copy this file to `raw_data/steam_matches_manual.json` and resolve the multi-candidate and missing entries by hand.


In [ ]:
output: dict[str, list[list]] = {}

for epic_name, candidates in match_results.items():
    output[epic_name] = [[appid, steam_name] for appid, steam_name, *_ in candidates]

OUT_FILE = DATA_DIR / "steam_matches.json"
with open(OUT_FILE, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"Saved {len(output)} entries to {OUT_FILE}")

# Summary
single = sum(1 for v in output.values() if len(v) == 1)
multi  = sum(1 for v in output.values() if len(v) > 1)
empty  = sum(1 for v in output.values() if len(v) == 0)
print(f"  Single match (done)   : {single}")
print(f"  Multiple candidates   : {multi}  ← needs manual review")
print(f"  No match              : {empty}  ← not found on Steam")